In [1]:
# Imports
from NavSysLib.utilities import *
from NavSysLib.Coords import *
from NavSysLib.Orbit import *

import numpy as np
from pathlib import Path

In [2]:
# Load and parse file
eph_path = Path("ns2025-2026_WA2.eph")
if not eph_path.exists():
    eph_path = Path("Tests/T2/ns2025-2026_WA2.eph")

wn = 2056

with eph_path.open("r", encoding="ascii") as f:
    eph_lines = [line.strip() for line in f if line.strip()]

orbits = [Orbit.from_eph_line(line, reference_wn=wn) for line in eph_lines]

print(f"Loaded {len(eph_lines)} lines from {eph_path}")
print(f"Parsed {len(orbits)} orbits")

Loaded 15 lines from ns2025-2026_WA2.eph
Parsed 15 orbits


Exercise 1

In [3]:
azores_coords = WGS84Coords.from_dms((37, 44, 20, 'N'), (25, 39, 10, 'W'), 0)
brazil_coords = WGS84Coords.from_dms((0, 22, 25, 'S'), (48, 7, 35, 'W'), 0)

loxodrome = azores_coords.loxodrome_to(brazil_coords, radius=6371000)
print(loxodrome)

0.7122150174169188
-0.006520790221865044
(np.float64(4827925.537325304), np.float64(-151.37720525562912))


Exercise 2

In [4]:
wn_end = 2057
tow_end = 50
dur_week = 2
dur_sec = 601790

wn_start, tow_start = calculate_start_time(wn_end, tow_end, dur_week, dur_sec)
print(f"WN: {wn_start}, TOW: {tow_start}")

WN: 2054, TOW: 3060


Exercise 3

In [5]:
eph_x = Ephemeris(wn=2058,
                  iode=20,
                  t_oe=0,
                  M_0=0.751,
                  e=0.0217,
                  sqrt_a=5154,
                  d_eta=0,
                  C_rc=0,
                  C_rs=0,
                  C_uc=0,
                  C_us=0,
                  C_ic=0,
                  C_is=0)
sat_x_orbit = Orbit(ephemeris=eph_x)

Exercise 4

In [6]:
svn2 = orbits[1]
print(svn2.ephemeris.sv_num)

print(f"Argument of perigee: {rad2deg(svn2.ephemeris.omega)}")
print(f"Inclination: {rad2deg(svn2.ephemeris.i_0)}")
print(f"Eccentricity: {(svn2.ephemeris.e)}")
print(f"Longitude of ascending node: {rad2deg(svn2.ephemeris.Omega_0)}")

2
Argument of perigee: -99.50491164836343
Inclination: 54.728876855213755
Eccentricity: 0.0188940634253
Longitude of ascending node: 178.04979558092552


Since argument of perigee is negative, it means that perigee is reached before the ascending node. As such, perigee is in the southern hemisphere, and thus apogee in the northern. Since the satellite travels slower around apogee, that means it spends the most time in the northern hemisphere.

Exercise 5

In [7]:
r = WGS84Coords.from_dm((38, 44.2523, 'N'), (9, 8.3104, 'W'), 195.4)
print(r.to_ecef_string())

wn_1 = 2056
tow_1 = 604790
wn_2 = 2057
tow_2 = 10

x=4918532.03 m, y=-791211.95 m, z=3969753.95 m


In [8]:
# a) Compute sat pos at t1 and t2
pos_1 = svn2.wgs84_ecef_position(wn_1, tow_1, return_coords=True)
print(pos_1.to_ecef_string())

pos_2 = svn2.wgs84_ecef_position(wn_2, tow_2, return_coords=True)
print(pos_2.to_ecef_string())

x=14470530.00 m, y=-14640714.38 m, z=17308528.45 m
x=14512314.79 m, y=-14643662.09 m, z=17269330.13 m


In [9]:
# b) Compute average speed of satellite

speed = svn2.get_inertial_speed_between_times(wn_1, tow_1, wn_2, tow_2)
print(speed)

3825.6554761571365


In [10]:
# c) Compute position at TX time

t_tx = svn2.get_tx_time_from_ref_point(wn_1, tow_1, r.to_ecef(), epsilon=1e-6)
print(t_tx)

pos_tx = svn2.get_pos_at_tx_time(t_tx, ref_wn=wn_1, ref_tow=tow_1, return_coords=True)
print(pos_tx.to_ecef_string())

print(t_tx)
print(t_tx//604800)
print(t_tx%604800)

Iteration 1: t_tx=1244073590.000000 s, s = [ 14470529.99864868 -14640714.37621673  17308528.18643447] m, d_Omega=0.000000000e+00 rad, d=0.000 m, d_prime=21470265.037 m
Iteration 2: t_tx=1244073589.928383 s, s = [ 14470303.74014983 -14640779.45024752  17308668.29525344] m, d_Omega=5.222401055e-06 rad, d=21470265.037 m, d_prime=21470293.399 m
Iteration 3: t_tx=1244073589.928383 s, s = [ 14470303.74004882 -14640779.45034734  17308668.29525344] m, d_Omega=5.222407953e-06 rad, d=21470293.399 m, d_prime=21470293.399 m
Iteration 4: t_tx=1244073589.928383 s, s = [ 14470303.74004882 -14640779.45034734  17308668.29525344] m, d_Omega=5.222407953e-06 rad, d=21470293.399 m, d_prime=21470293.399 m
Transmission time calculation converged in 4 iterations with final distance 21470293.399 m
1244073589.9283829
x=14470303.74 m, y=-14640779.45 m, z=17308668.56 m
1244073589.9283829
2056.0
604789.9283828735


In [11]:
# d) Compute direction cosines

cosines = r.direction_cosines_to(pos_1)
cosines_enu = r.direction_cosines_to_enu(pos_1)

print(f"ECEF cosines: {cosines}")
print(f"ENU cosines: {cosines_enu}")

e: -12156650.72772011, n: 3126777.8323788326, u: 17418708.083071876
ECEF cosines: (np.float64(0.4448942700713945), np.float64(-0.645055023546294), np.float64(0.6212673378307693))
ENU cosines: (np.float64(-0.5662086897282957), np.float64(0.1456329394662741), np.float64(0.8112945005474763))
